In [1]:
from pathlib import Path
import pandas as pd

In [2]:
PASTA_ENTRADA = Path("dados_implementacao")
HUB = "hub_implementacao.csv"
TRIBUNAL = "tribunal_implementacao.csv"
APRESENTACOES = "apresentacoes_implementacao.json"
PAGAMENTOS = "pagamentos_implementacao.json"

In [3]:
def limpar_processo(serie: pd.Series) -> pd.Series:
    """Mantém apenas os dígitos numéricos de uma coluna de processos."""
    return serie.fillna("").astype(str).str.replace(r"\D", "", regex=True)


def formatar_cnj(digitos: str) -> str:
    """Formata os 20 dígitos no padrão CNJ (NNNNNNN-DD.AAAA.J.TR.OOOO)."""
    limpo = "".join(ch for ch in str(digitos or "") if ch.isdigit())
    if len(limpo) == 20:
        return f"{limpo[:7]}-{limpo[7:9]}.{limpo[9:13]}.{limpo[13]}.{limpo[14:16]}.{limpo[16:20]}"
    return limpo


def converter_valor_serie(serie: pd.Series) -> pd.Series:
    """Converte coluna mista (float 15000.0, texto BR '2.500,50' e None) para float."""
    return pd.to_numeric(
        serie.astype(str)
        .str.replace(r"\.(?=\d{3})", "", regex=True)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def padronizar_data_pagamento(serie: pd.Series) -> pd.Series:
    """Padroniza datas 'AAAA-MM-DD' e 'DD/MM/AAAA' para 'AAAA-MM-DD'."""
    datas_iso = pd.to_datetime(serie, format="%Y-%m-%d", errors="coerce")
    datas_br = pd.to_datetime(serie, format="%d/%m/%Y", errors="coerce")
    return datas_iso.fillna(datas_br).dt.strftime("%Y-%m-%d")


def converter_data_brasilia(valor: str) -> pd.Timestamp:
    """Datas sem fuso já estão no dia civil de Brasília; datas com fuso são convertidas."""
    ts = pd.Timestamp(str(valor).strip())
    if ts.tzinfo is None:
        return ts.tz_localize("America/Sao_Paulo")
    return ts.tz_convert("America/Sao_Paulo")


def calcular_prazo_pagamento(serie_apresentado_em: pd.Series) -> pd.Series:
    """Aplica a regra constitucional de 2 de abril no fuso de Brasília."""
    datas_br = serie_apresentado_em.apply(converter_data_brasilia)
    ate_2_abril = (datas_br.dt.month < 4) | ((datas_br.dt.month == 4) & (datas_br.dt.day <= 2))
    ano_prazo = (datas_br.dt.year + 1).where(ate_2_abril, datas_br.dt.year + 2)
    return ano_prazo.astype(str) + "-12-31"

In [4]:
# Leitura dos arquivos
df_hub = pd.read_csv(PASTA_ENTRADA / HUB)
df_tribunal = pd.read_csv(PASTA_ENTRADA / TRIBUNAL, encoding="ISO-8859-1", sep=";")
df_apresentacoes = pd.read_json(PASTA_ENTRADA / APRESENTACOES)
df_pagamentos = pd.read_json(PASTA_ENTRADA / PAGAMENTOS)

In [5]:
# Tratamento do df_tribunal
df_tribunal["numero_processo"] = limpar_processo(df_tribunal["numero_processo"])
df_tribunal.insert(2, "valor_a_pagar", converter_valor_serie(df_tribunal["valor_face"]))
df_tribunal = df_tribunal.drop(columns=["valor_face"])

df_tribunal.head()

,numero_processo,ente_devedor,valor_a_pagar
0,00123451020198260053,Município de São Paulo,45000.0
1,00234562120208260053,Município de São Paulo,30000.0
2,00345673220188260405,Estado de São Paulo,12500.0
3,00456784320215020011,União,9800.0
4,00678906520168260053,Município de São Paulo,7000.0


In [6]:
# Tratamento do df_hub
df_hub.insert(0, "numero_processo", limpar_processo(df_hub["processo"]))
df_hub = df_hub.drop(columns=["processo"])
df_hub.insert(2, "valor_a_pagar_hub", df_hub["valor_face"].astype(float))
df_hub = df_hub.drop(columns=["valor_face"])
df_hub["prazo_hub_original"] = df_hub["prazo_pagamento"]
df_hub["prazo_pagamento"] = calcular_prazo_pagamento(df_hub["apresentado_em"])

df_hub.head()

,numero_processo,ente,valor_a_pagar_hub,apresentado_em,prazo_pagamento,prazo_hub_original
0,00123451020198260053,MUNICIPIO DE SAO PAULO,45000.0,2025-04-02,2026-12-31,2027-12-31
1,00234562120208260053,Município de São Paulo,31000.0,2025-04-03,2027-12-31,2027-12-31
2,00345673220188260405,Estado de São Paulo,12500.0,2024-02-29,2025-12-31,2025-12-31
3,00456784320215020011,Município de Santos,9800.0,2025-04-02T23:30:00-03:00,2026-12-31,2026-12-31
4,00567895420174036100,Estado de São Paulo,22000.0,2025-04-03T01:00:00Z,2026-12-31,2027-12-31


In [7]:
# Tratamento do df_apresentacoes
df_apresentacoes.insert(0, "numero_processo", limpar_processo(df_apresentacoes["processo"]))
df_apresentacoes = df_apresentacoes.drop(columns=["processo"])
df_apresentacoes["prazo_pagamento"] = calcular_prazo_pagamento(df_apresentacoes["apresentado_em"])
df_apresentacoes["apresentado_em"] = (
    df_apresentacoes["apresentado_em"].apply(converter_data_brasilia).dt.strftime("%Y-%m-%d")
)

df_apresentacoes

,numero_processo,apresentado_em,prazo_pagamento
0,00123451020198260053,2025-04-02,2026-12-31
1,00234562120208260053,2025-04-03,2027-12-31
2,00345673220188260405,2024-02-29,2025-12-31
3,00456784320215020011,2025-04-02,2026-12-31
4,00567895420174036100,2025-04-02,2026-12-31
5,00678906520168260053,2023-12-31,2025-12-31


In [8]:
# Tratamento do df_pagamentos
df_pagamentos.insert(0, "numero_processo", limpar_processo(df_pagamentos["processo"]))
df_pagamentos = df_pagamentos.drop(columns=["processo"])
df_pagamentos.insert(2, "valor_evento", converter_valor_serie(df_pagamentos["valor"]))
df_pagamentos = df_pagamentos.drop(columns=["valor"])
df_pagamentos["data_pagamento"] = padronizar_data_pagamento(df_pagamentos["data_pagamento"])

# Ordena por atualizado_em e mantém apenas a versão mais recente de cada evento_id
df_pagamentos["atualizado_em_ts"] = pd.to_datetime(df_pagamentos["atualizado_em"], utc=True)
df_pagamentos = df_pagamentos.sort_values(by="atualizado_em_ts")
df_pagamentos = df_pagamentos.drop_duplicates(subset=["evento_id"], keep="last")

# Separa eventos válidos dos eventos inválidos (quarentena)
condicao_valida = (
    (df_pagamentos["numero_processo"].str.len() == 20)
    & df_pagamentos["valor_evento"].notna()
    & (df_pagamentos["valor_evento"] > 0)
    & df_pagamentos["data_pagamento"].notna()
    & df_pagamentos["tipo"].isin(["pagamento", "estorno"])
)
df_pagamentos_descartados = df_pagamentos[~condicao_valida].drop(columns=["atualizado_em_ts"]).copy()
df_pagamentos_validos = df_pagamentos[condicao_valida].copy()

df_pagamentos_descartados["motivo_descarte"] = "Registro inválido"
df_pagamentos_descartados.loc[
    df_pagamentos_descartados["valor_evento"].isna(), "motivo_descarte"
] = "Valor nulo (null)"
df_pagamentos_descartados.loc[
    df_pagamentos_descartados["valor_evento"] <= 0, "motivo_descarte"
] = "Valor menor ou igual a zero"
df_pagamentos_descartados.loc[
    df_pagamentos_descartados["numero_processo"].str.len() != 20, "motivo_descarte"
] = "Número de processo vazio ou inválido"

# No estorno o impacto líquido no total pago é negativo (-valor_evento)
df_pagamentos_validos["valor_liquido"] = df_pagamentos_validos["valor_evento"].where(
    df_pagamentos_validos["tipo"] == "pagamento", -df_pagamentos_validos["valor_evento"]
)

df_pagamentos_validos

,numero_processo,evento_id,valor_evento,data_pagamento,tipo,atualizado_em,atualizado_em_ts,valor_liquido
6,00345673220188260405,ev-006,5000.0,2025-02-20,pagamento,2025-02-20T10:00:00-03:00,2025-02-20 13:00:00+00:00,5000.0
7,00345673220188260405,ev-007,5000.0,2025-02-21,estorno,2025-02-21T10:00:00-03:00,2025-02-21 13:00:00+00:00,-5000.0
8,00123451020198260053,ev-001,15500.0,2025-03-10,pagamento,2025-03-11T09:00:00-03:00,2025-03-11 12:00:00+00:00,15500.0
3,00123451020198260053,ev-002,2500.5,2025-04-10,pagamento,2025-04-10T09:00:00-03:00,2025-04-10 12:00:00+00:00,2500.5
2,00234562120208260053,ev-003,8000.0,2025-05-02,pagamento,2025-05-02T11:00:00-03:00,2025-05-02 14:00:00+00:00,8000.0
5,00234562120208260053,ev-005,1200.0,2025-06-01,pagamento,2025-06-01T10:00:00-03:00,2025-06-01 13:00:00+00:00,1200.0
9,00456784320215020011,ev-008,3000.0,2025-07-15,pagamento,2025-07-15T10:00:00-03:00,2025-07-15 13:00:00+00:00,3000.0
12,00678906520168260053,ev-011,4000.0,2025-08-10,pagamento,2025-08-10T10:00:00-03:00,2025-08-10 13:00:00+00:00,4000.0


In [9]:
# Criação da base de eventos válidos (LEFT JOIN + saldo acumulado) e resumo por precatório
df_pagamentos_validos = df_pagamentos_validos.sort_values(
    by=["numero_processo", "data_pagamento", "atualizado_em_ts"]
)
df_base = (
    df_pagamentos_validos.merge(
        df_apresentacoes[["numero_processo", "apresentado_em", "prazo_pagamento"]],
        on="numero_processo",
        how="left",
    ).merge(
        df_tribunal[["numero_processo", "ente_devedor", "valor_a_pagar"]],
        on="numero_processo",
        how="left",
    )
)
df_base["total_pago_acumulado"] = df_base.groupby("numero_processo")["valor_liquido"].cumsum()
df_base["valor_restante"] = df_base["valor_a_pagar"] - df_base["total_pago_acumulado"]

# Resumo consolidado por precatório
soma_por_processo = (
    df_pagamentos_validos.groupby("numero_processo", as_index=False)["valor_liquido"]
    .sum()
    .rename(columns={"valor_liquido": "total_pago"})
)
df_precatorios = (
    df_apresentacoes.merge(df_tribunal, on="numero_processo", how="outer")
    .merge(
        df_hub[["numero_processo", "ente", "valor_a_pagar_hub", "prazo_hub_original"]],
        on="numero_processo",
        how="outer",
    )
    .merge(soma_por_processo, on="numero_processo", how="left")
)
df_precatorios["total_pago"] = df_precatorios["total_pago"].fillna(0.0)
df_precatorios["ente_devedor"] = df_precatorios["ente_devedor"].fillna(df_precatorios["ente"])
df_precatorios["valor_a_pagar"] = df_precatorios["valor_a_pagar"].fillna(
    df_precatorios["valor_a_pagar_hub"]
)
df_precatorios["valor_restante"] = df_precatorios["valor_a_pagar"] - df_precatorios["total_pago"]

# Formata CNJ para evitar notação científica no Excel
df_base["numero_processo"] = df_base["numero_processo"].apply(formatar_cnj)
df_pagamentos_descartados["numero_processo"] = df_pagamentos_descartados[
    "numero_processo"
].apply(formatar_cnj)
df_precatorios["numero_processo"] = df_precatorios["numero_processo"].apply(formatar_cnj)

colunas_base = [
    "numero_processo",
    "evento_id",
    "ente_devedor",
    "tipo",
    "valor_a_pagar",
    "valor_evento",
    "valor_liquido",
    "total_pago_acumulado",
    "valor_restante",
    "apresentado_em",
    "prazo_pagamento",
    "data_pagamento",
    "atualizado_em",
]
df_base = df_base[colunas_base]

with pd.ExcelWriter("base_final.xlsx") as writer:
    df_base.to_excel(writer, sheet_name="pagamentos_validos", index=False)
    df_precatorios.to_excel(writer, sheet_name="precatorios_consolidados", index=False)
    df_pagamentos_descartados.to_excel(writer, sheet_name="eventos_descartados", index=False)

df_base

,numero_processo,evento_id,ente_devedor,tipo,valor_a_pagar,valor_evento,valor_liquido,total_pago_acumulado,valor_restante,apresentado_em,prazo_pagamento,data_pagamento,atualizado_em
0,0012345-10.2019.8.26.0053,ev-001,Município de São Paulo,pagamento,45000.0,15500.0,15500.0,15500.0,29500.0,2025-04-02,2026-12-31,2025-03-10,2025-03-11T09:00:00-03:00
1,0012345-10.2019.8.26.0053,ev-002,Município de São Paulo,pagamento,45000.0,2500.5,2500.5,18000.5,26999.5,2025-04-02,2026-12-31,2025-04-10,2025-04-10T09:00:00-03:00
2,0023456-21.2020.8.26.0053,ev-003,Município de São Paulo,pagamento,30000.0,8000.0,8000.0,8000.0,22000.0,2025-04-03,2027-12-31,2025-05-02,2025-05-02T11:00:00-03:00
3,0023456-21.2020.8.26.0053,ev-005,Município de São Paulo,pagamento,30000.0,1200.0,1200.0,9200.0,20800.0,2025-04-03,2027-12-31,2025-06-01,2025-06-01T10:00:00-03:00
4,0034567-32.2018.8.26.0405,ev-006,Estado de São Paulo,pagamento,12500.0,5000.0,5000.0,5000.0,7500.0,2024-02-29,2025-12-31,2025-02-20,2025-02-20T10:00:00-03:00
5,0034567-32.2018.8.26.0405,ev-007,Estado de São Paulo,estorno,12500.0,5000.0,-5000.0,0.0,12500.0,2024-02-29,2025-12-31,2025-02-21,2025-02-21T10:00:00-03:00
6,0045678-43.2021.5.02.0011,ev-008,União,pagamento,9800.0,3000.0,3000.0,3000.0,6800.0,2025-04-02,2026-12-31,2025-07-15,2025-07-15T10:00:00-03:00
7,0067890-65.2016.8.26.0053,ev-011,Município de São Paulo,pagamento,7000.0,4000.0,4000.0,4000.0,3000.0,2023-12-31,2025-12-31,2025-08-10,2025-08-10T10:00:00-03:00
